In [0]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os, time
import os, warnings, math, textwrap
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                         # non-interactive backend (safe for notebooks too)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LogNorm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# Spark / PySpark
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import DoubleType

In [0]:
# %pip install s3fs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 11.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.3/243.3 kB 14.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 54.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.5/152.5 kB 20.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.5/219.5 kB 27.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.4.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.10/site-packages, out

## Load the all Tables

In [1]:

# import s3fs
# DATA_ROOT = "s3://university-research-s20426"

# fs = s3fs.S3FileSystem(
#      key=
#     secret=
#     client_kwargs=
# )

# import glob
# cg  = spark.read.parquet(f"{DATA_ROOT}/MSCallGraph_clean")
# res  = spark.read.parquet(f"{DATA_ROOT}/resource")
# rtq  = spark.read.parquet(f"{DATA_ROOT}/MSRTQps_clean")




# TRAFFIC_COLS = [c for c in [
#     "providerRPC_MCR","providerRPC_RT",
#     "consumerRPC_MCR","consumerRPC_RT",
#     "HTTP_MCR","HTTP_RT",
#     "consumerMQ_MCR","consumerMQ_RT",
# ] if c in rtq.columns]
 
# tables = {
#     "MSResource":  (res,  ["cpu_utilization","memory_utilization",
#                             "timestamp","t_idx"]),
#     "MSRTQps":     (rtq,  TRAFFIC_COLS + ["timestamp","t_idx"]),
#     "MSCallGraph": (cg,  ['traceid','timestamp','rpcid','UM','rpctype','DM','interface','rt','t_idx']),
# }

In [0]:
# display(rtq.select(F.min("t_idx").alias("min_t_idx"), F.max("t_idx").alias("max_t_idx")))

In [0]:
# ── White Theme Configuration ────────────────────────────────────────────────
PALETTE = {
    "cpu":      "#E63946",
    "memory":   "#457B9D",
    "traffic":  "#2A9D8F",
    "rt":       "#E9C46A",
    "mcr":      "#F4A261",
    "callgraph":"#264653",
    "neutral":  "#A8DADC",
    "bg":       "#FFFFFF",   # changed to white
    "text":     "#000000",   # black text for contrast
    "grid":     "#DDDDDD",   # light gray grid
    "accent":   "#1D3557",   # deep blue accent
}

SEABORN_THEME = {
    "axes.facecolor":   PALETTE["bg"],
    "figure.facecolor": PALETTE["bg"],
    "text.color":       PALETTE["text"],
    "axes.labelcolor":  PALETTE["text"],
    "xtick.color":      PALETTE["text"],
    "ytick.color":      PALETTE["text"],
    "axes.edgecolor":   PALETTE["grid"],
    "grid.color":       PALETTE["grid"],
    "axes.grid":        True,
}
sns.set_theme(style="whitegrid", rc=SEABORN_THEME)

plt.rcParams.update({
    "font.family":      "monospace",
    "axes.titlesize":   13,
    "axes.titleweight": "bold",
    "axes.titlecolor":  PALETTE["accent"],
    "figure.dpi":       120,
    "savefig.bbox":     "tight",
    "savefig.facecolor":PALETTE["bg"],  # white background for saved figures
})

OUTPUT_DIR = "./eda_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, facecolor=PALETTE["bg"])
    plt.close(fig)
    print(f"saved → {path}")


In [0]:
res_df, _   = tables["MSResource"]
rtq_df, _   = tables["MSRTQps"]
cg_df, _    = tables["MSCallGraph"]

TRAFFIC_COLS_RT  = [c for c in TRAFFIC_COLS if c.endswith("_RT")]
TRAFFIC_COLS_MCR = [c for c in TRAFFIC_COLS if c.endswith("_MCR")]

## Dataset Overview

In [0]:
def _count(df): return df.count()
def _schema_info(df):
    return [(f.name, str(f.dataType)) for f in df.schema.fields]

with ThreadPoolExecutor(max_workers=3) as pool:
    f_res_cnt  = pool.submit(_count, res_df)
    f_rtq_cnt  = pool.submit(_count, rtq_df)
    f_cg_cnt   = pool.submit(_count, cg_df)

counts = {
    "MSResource":  f_res_cnt.result(),
    "MSRTQps":     f_rtq_cnt.result(),
    "MSCallGraph": f_cg_cnt.result(),
}
print("  Row counts:", counts)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Dataset Overview — Row Distribution & Schema", fontsize=15,
             color=PALETTE["accent"], fontweight="bold")

ax = axes[0]
labels = list(counts.keys())
sizes  = list(counts.values())
colors = [PALETTE["cpu"], PALETTE["traffic"], PALETTE["callgraph"]]
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, autopct="%1.1f%%",
    startangle=140, wedgeprops=dict(edgecolor=PALETTE["bg"], linewidth=2),
    textprops={"color": PALETTE["text"]},
)
for at in autotexts: at.set_fontsize(11)
ax.set_title("Row Count Distribution", color=PALETTE["accent"])

# Schema table
ax2 = axes[1]
ax2.axis("off")
schema_rows = []
for name, (df, _) in tables.items():
    for fname, ftype in _schema_info(df):
        schema_rows.append([name, fname, ftype])
schema_pd = pd.DataFrame(schema_rows, columns=["Table", "Column", "Type"])
tbl = ax2.table(
    cellText=schema_pd.values,
    colLabels=schema_pd.columns,
    cellLoc="left", loc="center",
)
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
for (r, c), cell in tbl.get_celld().items():
    cell.set_facecolor(PALETTE["grid"] if r == 0 else PALETTE["bg"])
    cell.set_text_props(color=PALETTE["text"])
    cell.set_edgecolor(PALETTE["grid"])
ax2.set_title("Schema Overview", color=PALETTE["accent"])
display(fig)

## MSResource

In [0]:
print("\n[B] MSResource — CPU & Memory analysis …")

# ── Parallel stats collection ──────────────────────────────────────────────────
def _res_global_stats():
    return (res_df.select(
        F.mean("cpu_utilization").alias("cpu_mean"),
        F.stddev("cpu_utilization").alias("cpu_std"),
        F.percentile_approx("cpu_utilization", [0.25, 0.5, 0.75, 0.95, 0.99]).alias("cpu_pct"),
        F.mean("memory_utilization").alias("mem_mean"),
        F.stddev("memory_utilization").alias("mem_std"),
        F.percentile_approx("memory_utilization", [0.25, 0.5, 0.75, 0.95, 0.99]).alias("mem_pct"),
    ).collect()[0])

def _res_histogram(col, n_bins=50):
    """Compute histogram fully in Spark via bucket counts."""
    bounds = res_df.select(F.min(col), F.max(col)).collect()[0]
    lo, hi = float(bounds[0]), float(bounds[1])
    step   = (hi - lo) / n_bins
    return (res_df
        .withColumn("bucket", ((F.col(col) - lo) / step).cast("int"))
        .withColumn("bucket", F.least(F.col("bucket"), F.lit(n_bins - 1)))
        .groupBy("bucket").count()
        .orderBy("bucket")
        .toPandas()
        .assign(bin_center=lambda d: lo + (d["bucket"] + 0.5) * step)
    ), lo, hi, step

def _res_timeseries_agg():
    """Hourly mean CPU & memory per t_idx bucket (no sampling)."""
    return (res_df
        .groupBy("t_idx")
        .agg(
            F.mean("cpu_utilization").alias("cpu_mean"),
            F.mean("memory_utilization").alias("mem_mean"),
            F.stddev("cpu_utilization").alias("cpu_std"),
            F.stddev("memory_utilization").alias("mem_std"),
           F.count("*").alias("n"),
        )
        .orderBy("t_idx")
        .toPandas()
    )

def _res_per_service_stats():
    """Top-N services by mean CPU (for box-like spread)."""
    svc_col = "msname" if "msname" in res_df.columns else \
              "service" if "service" in res_df.columns else None
    if svc_col is None:
        return None, None
    cpu_svc = (res_df
        .groupBy(svc_col)
        .agg(
            F.mean("cpu_utilization").alias("cpu_mean"),
            F.stddev("cpu_utilization").alias("cpu_std"),
            F.percentile_approx("cpu_utilization", [0.25, 0.5, 0.75]).alias("cpu_pct"),
            F.count("*").alias("n"),
        )
        .orderBy(F.desc("n"))
        .limit(10)
        .toPandas()
    )
    return cpu_svc, svc_col

def _res_outliers(col, k=3.0):
    """IQR-based outlier count per t_idx."""
    q = res_df.select(
        F.percentile_approx(col, 0.25).alias("q1"),
        F.percentile_approx(col, 0.75).alias("q3"),
    ).collect()[0]
    iqr = q["q3"] - q["q1"]
    lo, hi = q["q1"] - k * iqr, q["q3"] + k * iqr
    return (res_df
        .withColumn("is_outlier", (F.col(col) < lo) | (F.col(col) > hi))
        .groupBy("t_idx")
        .agg(
            F.sum(F.col("is_outlier").cast("int")).alias("outlier_count"),
            F.count("*").alias("total"),
        )
        .orderBy("t_idx")
        .toPandas()
        .assign(outlier_rate=lambda d: d["outlier_count"] / d["total"])
    )

def _res_joint_hist():
    """2-D histogram: cpu vs memory (spark bucketize both)."""
    n = 40
    bounds = res_df.select(
        F.min("cpu_utilization"), F.max("cpu_utilization"),
        F.min("memory_utilization"), F.max("memory_utilization"),
    ).collect()[0]
    clo, chi = float(bounds[0]), float(bounds[1])
    mlo, mhi = float(bounds[2]), float(bounds[3])
    cstep = (chi - clo) / n
    mstep = (mhi - mlo) / n
    return (res_df
        .withColumn("cb", F.least(((F.col("cpu_utilization") - clo) / cstep).cast("int"), F.lit(n-1)))
        .withColumn("mb", F.least(((F.col("memory_utilization") - mlo) / mstep).cast("int"), F.lit(n-1)))
        .groupBy("cb", "mb").count()
        .toPandas()
    ), clo, chi, mlo, mhi, n

with ThreadPoolExecutor(max_workers=6) as pool:
    f_gstats   = pool.submit(_res_global_stats)
    f_cpu_hist = pool.submit(_res_histogram, "cpu_utilization")
    f_mem_hist = pool.submit(_res_histogram, "memory_utilization")
    f_ts_agg   = pool.submit(_res_timeseries_agg)
    f_svc      = pool.submit(_res_per_service_stats)
    f_jh       = pool.submit(_res_joint_hist)
    f_cpu_out  = pool.submit(_res_outliers, "cpu_utilization")
    f_mem_out  = pool.submit(_res_outliers, "memory_utilization")

gstats          = f_gstats.result()
cpu_hist_pd, cpu_lo, cpu_hi, cpu_step = f_cpu_hist.result()
mem_hist_pd, mem_lo, mem_hi, mem_step = f_mem_hist.result()
ts_agg_pd       = f_ts_agg.result()
svc_pd, svc_col = f_svc.result()
jh_pd, clo, chi, mlo, mhi, jn = f_jh.result()
cpu_out_pd      = f_cpu_out.result()
mem_out_pd      = f_mem_out.result()


####  CPU & Memory analysis

In [0]:

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("MSResource — CPU & Memory Utilization Distribution\n",color=PALETTE["accent"])

for ax, hist_pd, col, color, label, glo, ghi in [
    (axes[0], cpu_hist_pd, "cpu_utilization", PALETTE["cpu"],
     "CPU Utilization", cpu_lo, cpu_hi),
    (axes[1], mem_hist_pd, "memory_utilization", PALETTE["memory"],
     "Memory Utilization", mem_lo, mhi),
]:
    ax.bar(hist_pd["bin_center"], hist_pd["count"],
           width=(ghi - glo) / len(hist_pd) * 0.9,
           color=color, alpha=0.85, edgecolor=PALETTE["bg"])
    ax.set_xlabel(label, color=PALETTE["text"])
    ax.set_ylabel("Count", color=PALETTE["text"])
    ax.set_title(f"{label} Histogram")
 

    #annotate stats
    pcts = gstats["cpu_pct"] if "cpu" in label.lower() else gstats["mem_pct"]
    mean = gstats["cpu_mean"] if "cpu" in label.lower() else gstats["mem_mean"]
    for p, lbl in zip(pcts[1::2], ["p50","p75","p99"]):
        ax.axvline(p, color="white", lw=0.8, linestyle="--", alpha=0.5)
        ax.text(p, ax.get_ylim()[1]*0.9, lbl, color="white",
                fontsize=7, rotation=90, va="top", ha="right")
    ax.axvline(mean, color=PALETTE["accent"], lw=1.5, linestyle="-")
    ax.text(mean, ax.get_ylim()[1]*0.95, f"μ={mean:.3f}",
            color=PALETTE["accent"], fontsize=8, ha="left")
save(fig, "cpu_memory_analysis.png")
display(fig)

#### CPU vs Memory Utilization — 2D Density Heatmap

In [0]:
fig, ax = plt.subplots(figsize=(9, 7))
fig.suptitle("CPU vs Memory Utilization — 2D Density Heatmap\n", color=PALETTE["accent"])

pivot = jh_pd.pivot_table(index="mb", columns="cb", values="count", fill_value=0)
pivot_arr = pivot.values.astype(float)
im = ax.imshow(
    pivot_arr + 1,          # +1 to avoid log(0)
    aspect="auto", origin="lower",
    norm=LogNorm(vmin=1, vmax=pivot_arr.max()),
    cmap="plasma",
    extent=[clo, chi, mlo, mhi],
)
ax.set_xlabel("CPU Utilization", color=PALETTE["text"])
ax.set_ylabel("Memory Utilization", color=PALETTE["text"])
ax.set_title("Density (log scale) — GNN Node Feature Correlation")
cb = fig.colorbar(im, ax=ax, label="Count (log)")
cb.ax.yaxis.label.set_color(PALETTE["text"])
cb.ax.tick_params(colors=PALETTE["text"])

# Pearson r from bucket-level data (weighted)
flat = jh_pd.copy()
flat["cpu_c"] = clo + (flat["cb"] + 0.5) * (chi - clo) / jn
flat["mem_c"] = mlo + (flat["mb"] + 0.5) * (mhi - mlo) / jn
r = np.corrcoef(
    np.repeat(flat["cpu_c"], flat["count"].clip(upper=1000)),
    np.repeat(flat["mem_c"], flat["count"].clip(upper=1000))
)[0, 1]
ax.text(0.05, 0.95, f"Pearson r ≈ {r:.3f}", transform=ax.transAxes,
        color="white", fontsize=10, va="top",
        bbox=dict(facecolor=PALETTE["grid"], alpha=0.7, boxstyle="round"))
save(fig, "cpu_memory_heatmap.png")
display(fig)


#### Time-series with seasonality bands 

In [0]:
fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True)
fig.suptitle("MSResource — Temporal Trends ",
             color=PALETTE["accent"])

for ax, col, color, label in [
    (axes[0], "cpu_mean",  PALETTE["cpu"],    "Mean CPU Utilization"),
    (axes[1], "mem_mean",  PALETTE["memory"], "Mean Memory Utilization"),
]:
    std_col = col.replace("mean", "std")
    ax.plot(ts_agg_pd["t_idx"], ts_agg_pd[col], color=color, lw=1.2, label=label)
    ax.fill_between(
        ts_agg_pd["t_idx"],
        ts_agg_pd[col] - ts_agg_pd[std_col].fillna(0),
        ts_agg_pd[col] + ts_agg_pd[std_col].fillna(0),
        alpha=0.2, color=color, label="±1 std",
    )
    # Simple rolling seasonality window (7-period)
    roll = ts_agg_pd[col].rolling(7, center=True).mean()
    ax.plot(ts_agg_pd["t_idx"], roll, color="white", lw=0.8,
            linestyle="--", alpha=0.6, label="7-step rolling avg")
    ax.set_ylabel(label, color=PALETTE["text"])
    ax.legend(loc="upper right", fontsize=8,
              facecolor=PALETTE["grid"], labelcolor=PALETTE["text"])

axes[1].set_xlabel("t_idx (temporal index)", color=PALETTE["text"])
save(fig, "Time-series_seasonality.png")
display(fig)

#### MSResource — Per-Service CPU & Memory Spread  (Top 10 by Record Volume)


In [0]:
"""
B5 — Per-Service CPU & Memory Spread (Top 10 by Volume)
Anonymized service labels, parallel Spark stats collection.
"""

# ── Spark stats (parallel) ────────────────────────────────────────────────────
def _res_per_service_stats(metric_col):
    svc_col = next(
        (c for c in ["msname", "service", "serviceName", "nodeid"]
         if c in res_df.columns), None
    )
    if svc_col is None:
        return None, None
    df = (res_df
        .groupBy(svc_col)
        .agg(
            F.mean(metric_col).alias("mean"),
            F.stddev(metric_col).alias("std"),
            F.percentile_approx(metric_col, [0.25, 0.5, 0.75]).alias("pct"),
            F.count("*").alias("n"),
        )
        .orderBy(F.desc("n"))
        .limit(10)
        .toPandas()
    )
    return df, svc_col

with ThreadPoolExecutor(max_workers=2) as pool:
    f_cpu = pool.submit(_res_per_service_stats, "cpu_utilization")
    f_mem = pool.submit(_res_per_service_stats, "memory_utilization")

cpu_svc_pd, svc_col = f_cpu.result()
mem_svc_pd, _       = f_mem.result()


# ── Helper: draw one pair of axes (spread + scatter) ─────────────────────────
def _plot_service_spread(fig, axes, svc_pd, svc_col, metric_label, bar_color):
    """
    axes[0] — horizontal IQR bar chart (anonymized labels)
    axes[1] — scatter: record count vs mean, coloured by std dev
    """
    svc_pd  = svc_pd.sort_values("mean", ascending=False).reset_index(drop=True)
    n_svc   = len(svc_pd)
    anon    = [f"Top {i+1} service" for i in range(n_svc)]
    means   = svc_pd["mean"].values
    stds    = svc_pd["std"].fillna(0).values
    pcts    = np.array([list(p) for p in svc_pd["pct"]])   # (n, 3): Q1, med, Q3
    y       = np.arange(n_svc)

    # ── left: IQR spread ──────────────────────────────────────────────────────
    ax = axes[0]
    ax.barh(
        y, pcts[:, 2] - pcts[:, 0], left=pcts[:, 0],
        color=bar_color, alpha=0.55, label="IQR (Q1–Q3)",
    )
    # whiskers to ±1.5 IQR proxied by std (no raw data needed)
    iqr = pcts[:, 2] - pcts[:, 0]
    lo_w = np.maximum(pcts[:, 0] - 1.5 * iqr, 0)
    hi_w = pcts[:, 2] + 1.5 * iqr
    for i in range(n_svc):
        ax.plot([lo_w[i], hi_w[i]], [i, i],
                color=bar_color, lw=0.8, alpha=0.4)
        ax.plot([lo_w[i], lo_w[i]], [i - 0.15, i + 0.15],
                color=bar_color, lw=1.2, alpha=0.5)
        ax.plot([hi_w[i], hi_w[i]], [i - 0.15, i + 0.15],
                color=bar_color, lw=1.2, alpha=0.5)

    ax.scatter(pcts[:, 1], y, color=PALETTE["accent"],
               s=50, marker="|", zorder=5, linewidths=2, label="Median")
    ax.scatter(means, y, color="white",
               s=40, zorder=6, label="Mean")

    ax.set_yticks(y)
    ax.set_yticklabels(anon, fontsize=8)
    ax.set_xlabel(metric_label, color=PALETTE["text"])
    ax.set_title(f"{metric_label} Spread per Service\n(IQR box ·  median | · mean ●)")
    ax.legend(fontsize=8, facecolor=PALETTE["grid"],
              labelcolor=PALETTE["text"], loc="upper right")

    # annotate mean values
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(
            ax.get_xlim()[1] * 0.99, i,
            f"{m:.3f} ± {s:.3f}",
            va="center", ha="right", fontsize=6.5,
            color=PALETTE["text"], alpha=0.75,
        )

    # ── right: volume vs mean scatter ────────────────────────────────────────
    ax2 = axes[1]
    sc  = ax2.scatter(
        svc_pd["n"], means,
        c=stds, cmap="plasma",
        s=80, alpha=0.9, edgecolors=PALETTE["bg"], linewidths=0.5,
    )
    cb = fig.colorbar(sc, ax=ax2, label=f"{metric_label} Std Dev", pad=0.02)
    cb.ax.yaxis.label.set_color(PALETTE["text"])
    cb.ax.tick_params(colors=PALETTE["text"])

    for i, row in svc_pd.iterrows():
        ax2.annotate(
            anon[i],
            (row["n"], row["mean"]),
            xytext=(4,3), textcoords="offset points",
            fontsize=7, color=PALETTE["text"], alpha=0.85,
        )

    ax2.set_xlabel("Record Count  (node data volume)", color=PALETTE["text"])
    ax2.set_ylabel(f"Mean {metric_label}", color=PALETTE["text"])
    ax2.set_title(
        f"Service Volume vs {metric_label}\n"
    )


# ── Figure: 2 rows × 2 cols ───────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.suptitle(
    "MSResource — Per-Service CPU & Memory Spread  (Top 10 by Record Volume)\n",
    color=PALETTE["accent"], fontsize=14, fontweight="bold",
)

gs = gridspec.GridSpec(
    2, 2, figure=fig,
    hspace=0.45, wspace=0.35,
    left=0.08, right=0.96, top=0.91, bottom=0.06,
)

axes_cpu = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
axes_mem = [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]

if cpu_svc_pd is not None:
    _plot_service_spread(
        fig, axes_cpu, cpu_svc_pd, svc_col,
        metric_label="CPU Utilization",
        bar_color=PALETTE["cpu"],
    )
else:
    for ax in axes_cpu:
        ax.text(0.5, 0.5, "No service column found",
                ha="center", va="center", color=PALETTE["text"],
                transform=ax.transAxes)

if mem_svc_pd is not None:
    _plot_service_spread(
        fig, axes_mem, mem_svc_pd, svc_col,
        metric_label="Memory Utilization",
        bar_color=PALETTE["memory"],
    )
else:
    for ax in axes_mem:
        ax.text(0.5, 0.5, "No service column found",
                ha="center", va="center", color=PALETTE["text"],
                transform=ax.transAxes)

# row labels
for row_ax, label, color in [
    (axes_cpu[0], "CPU", PALETTE["cpu"]),
    (axes_mem[0], "MEM", PALETTE["memory"]),
]:
    row_ax.annotate(
        label,
        xy=(-0.18, 0.5), xycoords="axes fraction",
        fontsize=18, fontweight="bold", color=color,
        rotation=90, va="center", ha="center",
    )
save(fig, "msresource_spread.png")
display(fig)
plt.close(fig)

## MSRTQps Table

In [0]:

print("\n[C] MSRTQps — Traffic (MCR / RT) analysis …")

def _rtq_global_stats():
    agg_exprs = []
    for c in TRAFFIC_COLS:
        agg_exprs += [
            F.mean(c).alias(f"{c}_mean"),
            F.stddev(c).alias(f"{c}_std"),
            F.percentile_approx(c, [0.5, 0.95, 0.99]).alias(f"{c}_pct"),
        ]
    return rtq_df.select(*agg_exprs).collect()[0]

def _rtq_timeseries():
    agg_exprs = [F.mean(c).alias(c) for c in TRAFFIC_COLS]
    return (rtq_df
        .groupBy("t_idx")
        .agg(*agg_exprs)
        .orderBy("t_idx")
        .toPandas()
    )

def _rtq_histogram(col, n_bins=40):
    bounds = rtq_df.select(
        F.percentile_approx(col, 0.01).alias("lo"),
        F.percentile_approx(col, 0.99).alias("hi"),
    ).collect()[0]
    lo, hi = float(bounds["lo"]), float(bounds["hi"])
    step = (hi - lo) / n_bins
    return (rtq_df
        .filter((F.col(col) >= lo) & (F.col(col) <= hi))
        .withColumn("bucket", ((F.col(col) - lo) / step).cast("int"))
        .withColumn("bucket", F.least(F.col("bucket"), F.lit(n_bins - 1)))
        .groupBy("bucket").count()
        .orderBy("bucket")
        .toPandas()
        .assign(bin_center=lambda d: lo + (d["bucket"] + 0.5) * step)
    ), lo, hi

def _rtq_corr():
    """Pairwise correlation via Spark."""
    if len(TRAFFIC_COLS) < 2:
        return None
    # Use Spark's correlation (column-by-column, pairwise)
    pairs = {}
    for i, c1 in enumerate(TRAFFIC_COLS):
        for c2 in TRAFFIC_COLS[i:]:
            key = (c1, c2)
            pairs[key] = rtq_df.select(F.corr(c1, c2).alias("r")).collect()[0]["r"]
    return pairs

with ThreadPoolExecutor(max_workers=4) as pool:
    f_rtq_stats = pool.submit(_rtq_global_stats)
    f_rtq_ts    = pool.submit(_rtq_timeseries)
    f_rtq_corr  = pool.submit(_rtq_corr)
    hist_futures = {
        col: pool.submit(_rtq_histogram, col) for col in TRAFFIC_COLS
    }

rtq_stats  = f_rtq_stats.result()
rtq_ts_pd  = f_rtq_ts.result()
rtq_corr   = f_rtq_corr.result()
rtq_hists  = {col: f.result() for col, f in hist_futures.items()}




#### Traffic histograms grid

In [0]:
# ── C1. Traffic histograms grid 
n_cols = 4
n_rows = math.ceil(len(TRAFFIC_COLS) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
fig.suptitle("Traffic Column Distributions ",
             color=PALETTE["accent"], fontsize=13)
axes_flat = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()

for i, col in enumerate(TRAFFIC_COLS):
    ax  = axes_flat[i]
    hist_pd, lo, hi = rtq_hists[col]
    color = PALETTE["rt"] if col.endswith("_RT") else PALETTE["mcr"]
    ax.bar(hist_pd["bin_center"], hist_pd["count"],
           width=(hi - lo) / len(hist_pd) * 0.9,
           color=color, alpha=0.85, edgecolor=PALETTE["bg"])
    mean_v = rtq_stats[f"{col}_mean"]
    p95_v  = rtq_stats[f"{col}_pct"][1]
    if mean_v is not None:
        ax.axvline(mean_v, color="white", lw=1, linestyle="--", alpha=0.7)
    if p95_v is not None:
        ax.axvline(p95_v, color=PALETTE["accent"], lw=1, linestyle=":", alpha=0.7)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("Value", fontsize=7)
    ax.tick_params(labelsize=6)

# hide unused axes
for j in range(len(TRAFFIC_COLS), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
save(fig, "Traffic_histograms_grid.png")
display(fig)


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import NumericType
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_boxplot_stats(name, df, num_cols):
    """Function to be executed in parallel for each table."""
    schema_map = {f.name: f.dataType for f in df.schema.fields}
    valid_cols = [
        c for c in num_cols 
        if c in df.columns and isinstance(schema_map.get(c), NumericType)
    ]
    
    if not valid_cols:
        return name, None

    # 1. Ask Spark to compute the 5-number summary in parallel
    stat_exprs = []
    for col in valid_cols:
        stat_exprs += [
            F.min(col).alias(f"{col}__min"),
            F.percentile_approx(col, 0.25).alias(f"{col}__q1"),
            F.percentile_approx(col, 0.50).alias(f"{col}__med"),
            F.percentile_approx(col, 0.75).alias(f"{col}__q3"),
            F.max(col).alias(f"{col}__max")
        ]

    # Collect only a single row of stats back to the driver
    try:
        stats_row = df.agg(*stat_exprs).collect()[0]
    except Exception as e:
        print(f"Error computing stats for {name}: {e}")
        return name, None

    # 2. Format the stats for Matplotlib's bxp function
    bxp_stats = []
    for col in valid_cols:
        c_min = stats_row[f"{col}__min"]
        c_q1 = stats_row[f"{col}__q1"]
        c_med = stats_row[f"{col}__med"]
        c_q3 = stats_row[f"{col}__q3"]
        c_max = stats_row[f"{col}__max"]

        if c_q1 is None or c_q3 is None:
            continue
            
        iqr = c_q3 - c_q1
        
        if iqr == 0:
            continue

        # Using your multiplier of 3 * IQR for the extreme outlier boundary
        # We cap the whiskers at the actual min/max so they don't extend past the data
        whislo = max(c_min, c_q1 - 3 * iqr) if c_min is not None else c_q1
        whishi = min(c_max, c_q3 + 3 * iqr) if c_max is not None else c_q3

        bxp_stats.append({
            'label': col,
            'med': c_med,
            'q1': c_q1,
            'q3': c_q3,
            'whislo': whislo,
            'whishi': whishi,
            'fliers': [] # Deliberately empty: plotting millions of individual dots causes crashes
        })

    return name, bxp_stats

# --- 3. Parallel Execution via ThreadPoolExecutor ---
# Adjust max_workers based on the size of your Spark cluster. 
# 3-5 is usually safe without overwhelming the Spark driver.
max_concurrent_tables = 4 
all_plot_data = {}

print(f"Starting parallel processing with {max_concurrent_tables} workers...")

with ThreadPoolExecutor(max_workers=max_concurrent_tables) as executor:
    # Submit all tables to be processed
    futures = {
        executor.submit(get_boxplot_stats, name, t[0], t[1]): name 
        for name, t in tables.items()
    }
    
    # Gather results as they finish
    for future in as_completed(futures):
        table_name, stats = future.result()
        if stats:
            all_plot_data[table_name] = stats
            print(f"Finished computing stats for: {table_name}")

# --- 4. Plotting (Fast & Memory Safe) ---
for table_name, stats in all_plot_data.items():
    if not stats:
        continue
        
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # ax.bxp draws standard boxplots from pre-computed dictionaries
    ax.bxp(stats, showfliers=False, widths=0.5, patch_artist=True)
    
    ax.set_title(f"Box Plot of Numeric Columns: {table_name}", fontsize=14)
    ax.set_ylabel("Values")
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Rotate labels if there are many columns
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

####   Traffic time-series (Seasonality Analysis)


In [0]:
if len(TRAFFIC_COLS) > 0 and not rtq_ts_pd.empty:
    fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True)
    fig.suptitle("MSRTQps — MCR & RT Temporal Patterns\n"
                 "Seasonality analysis for temporal GNN window sizing",
                 color=PALETTE["accent"])

    ax_mcr, ax_rt = axes[0], axes[1]
    for col in TRAFFIC_COLS_MCR:
        if col in rtq_ts_pd.columns:
            ax_mcr.plot(rtq_ts_pd["t_idx"], rtq_ts_pd[col], lw=1.0, alpha=0.8, label=col)
    for col in TRAFFIC_COLS_RT:
        if col in rtq_ts_pd.columns:
            ax_rt.plot(rtq_ts_pd["t_idx"], rtq_ts_pd[col], lw=1.0, alpha=0.8, label=col)

    ax_mcr.set_ylabel("Mean Call Rate (MCR)"); ax_mcr.legend(fontsize=7, facecolor=PALETTE["grid"], labelcolor=PALETTE["text"])
    ax_rt.set_ylabel("Mean Response Time (RT)"); ax_rt.legend(fontsize=7, facecolor=PALETTE["grid"], labelcolor=PALETTE["text"])
    ax_rt.set_xlabel("t_idx")
    save(fig,"Traffic_time_series.png")
    display(fig)
    

#### Seasonality Analysis using ACF

In [0]:
# ── C4. Seasonality decomposition proxy (ACF via pure pandas/numpy) ───────────
print(" Computing ACF for seasonality …")

if not rtq_ts_pd.empty and TRAFFIC_COLS_MCR:
    # Get ALL available MCR columns instead of just the first one
    valid_mcr_cols = [c for c in TRAFFIC_COLS_MCR if c in rtq_ts_pd.columns]
    
    if not valid_mcr_cols:
        print("No valid MCR columns found in the dataframe.")
        
    for mcr_col in valid_mcr_cols:
        series = rtq_ts_pd[mcr_col].ffill().bfill().values
        n      = len(series)
        
        if n < 2:
            continue  # Skip if not enough data
            
        max_lag = min(n // 2, 200)
        series_demean = series - series.mean()
        
        # Calculate variance to prevent division by zero on flatlines
        c0 = np.dot(series_demean, series_demean) / n
        if c0 == 0:
            print(f"Skipping {mcr_col}: series has zero variance (constant values).")
            continue
            
        acf    = [1.0] + [
            np.dot(series_demean[:n-k], series_demean[k:]) / (n * c0)
            for k in range(1, max_lag + 1)
        ]
        acf    = np.array(acf)
        ci     = 1.96 / np.sqrt(n)

        fig, axes = plt.subplots(2, 1, figsize=(16, 8))
        fig.suptitle(f"MSRTQps — Seasonality Patterns ({mcr_col})\n",
                     color=PALETTE["accent"])

        # Top Plot: ACF
        ax = axes[0]
        ax.bar(range(max_lag + 1), acf, color=PALETTE["mcr"], alpha=0.7, width=0.8)
        ax.axhline(ci, color="white", lw=0.8, linestyle="--", alpha=0.6, label=f"+1.96/√n = {ci:.3f}")
        ax.axhline(-ci, color="white", lw=0.8, linestyle="--", alpha=0.6)
        ax.axhline(0, color=PALETTE["grid"], lw=0.5)
        ax.set_xlabel("Lag"); ax.set_ylabel("Autocorrelation")
        ax.set_title(f"ACF  Identify dominant seasonal period ({mcr_col})")
        ax.legend(fontsize=8, facecolor=PALETTE["grid"], labelcolor=PALETTE["text"])

        # Bottom Plot: Rolling std as volatility proxy
        window = 10
        roll_std = pd.Series(series).rolling(window).std().values
        ax2 = axes[1]
        ax2.fill_between(range(n), roll_std, color=PALETTE["mcr"], alpha=0.6)
        ax2.set_xlabel("t_idx"); ax2.set_ylabel(f"Rolling Std (window={window})")
        ax2.set_title(f"Traffic Volatility  in  ({mcr_col})")
        
        plt.tight_layout()
        save(fig,"seasonality_acf.png")
        display(fig)
        plt.close(fig) 

## MSCallGraph

In [0]:

def _cg_rpctype_dist():
    return (cg_df.groupBy("rpctype").count()
            .orderBy(F.desc("count")).toPandas())

def _cg_rt_histogram(n_bins=50):
    bounds = cg_df.select(
        F.percentile_approx("rt", 0.01).alias("lo"),
        F.percentile_approx("rt", 0.99).alias("hi"),
    ).collect()[0]
    lo, hi = float(bounds["lo"]), float(bounds["hi"])
    step   = (hi - lo) / n_bins
    return (cg_df
        .filter((F.col("rt") >= lo) & (F.col("rt") <= hi))
        .withColumn("bucket", ((F.col("rt") - lo) / step).cast("int"))
        .withColumn("bucket", F.least(F.col("bucket"), F.lit(n_bins - 1)))
        .groupBy("bucket").count().orderBy("bucket")
        .toPandas()
        .assign(bin_center=lambda d: lo + (d["bucket"] + 0.5) * step)
    ), lo, hi

def _cg_top_edges():
    """Top 30 UM→DM pairs by call volume (edge weights)."""
    return (cg_df
        .groupBy("UM", "DM")
        .agg(
            F.count("*").alias("call_count"),
            F.mean("rt").alias("mean_rt"),
            F.percentile_approx("rt", 0.95).alias("p95_rt"),
        )
        .orderBy(F.desc("call_count"))
        .limit(30)
        .toPandas()
    )

def _cg_node_in_out():
    """In-degree and out-degree per microservice node."""
    out_deg = (cg_df.groupBy("UM").agg(F.count("*").alias("out_degree")).toPandas()
               .rename(columns={"UM": "node"}))
    in_deg  = (cg_df.groupBy("DM").agg(F.count("*").alias("in_degree")).toPandas()
               .rename(columns={"DM": "node"}))
    return pd.merge(out_deg, in_deg, on="node", how="outer").fillna(0)

def _cg_rt_per_interface():
    """Top 20 interfaces by mean RT."""
    return (cg_df
        .groupBy("interface")
        .agg(
            F.mean("rt").alias("mean_rt"),
            F.percentile_approx("rt", [0.25, 0.5, 0.75]).alias("pct_rt"),
            F.count("*").alias("n"),
        )
        .orderBy(F.desc("n"))
        .limit(20)
        .toPandas()
    )

def _cg_rt_by_rpctype():
    """RT distribution stats per rpctype."""
    return (cg_df
        .groupBy("rpctype")
        .agg(
            F.percentile_approx("rt", [0.25, 0.5, 0.75, 0.95]).alias("pct"),
            F.mean("rt").alias("mean"),
            F.count("*").alias("n"),
        )
        .toPandas()
    )

def _cg_timeseries():
    return (cg_df
        .groupBy("t_idx")
        .agg(
            F.count("*").alias("call_count"),
            F.mean("rt").alias("mean_rt"),
            F.countDistinct("traceid").alias("unique_traces"),
        )
        .orderBy("t_idx")
        .toPandas()
    )

def _cg_outlier_rt():
    q = cg_df.select(
        F.percentile_approx("rt", 0.25).alias("q1"),
        F.percentile_approx("rt", 0.75).alias("q3"),
    ).collect()[0]
    iqr = q["q3"] - q["q1"]
    hi  = q["q3"] + 3.0 * iqr
    return (cg_df
        .withColumn("outlier", (F.col("rt") > hi).cast("int"))
        .groupBy("t_idx")
        .agg(
            F.sum("outlier").alias("outlier_count"),
            F.count("*").alias("total"),
        )
        .orderBy("t_idx")
        .toPandas()
        .assign(rate=lambda d: d["outlier_count"] / d["total"])
    )

with ThreadPoolExecutor(max_workers=7) as pool:
    f_rpctype    = pool.submit(_cg_rpctype_dist)
    f_rt_hist    = pool.submit(_cg_rt_histogram)
    f_top_edges  = pool.submit(_cg_top_edges)
    f_node_deg   = pool.submit(_cg_node_in_out)
    f_iface      = pool.submit(_cg_rt_per_interface)
    f_rt_rpc     = pool.submit(_cg_rt_by_rpctype)
    f_cg_ts      = pool.submit(_cg_timeseries)
    f_rt_out     = pool.submit(_cg_outlier_rt)

rpctype_pd   = f_rpctype.result()
rt_hist_pd, rt_lo, rt_hi = f_rt_hist.result()
top_edges_pd = f_top_edges.result()
node_deg_pd  = f_node_deg.result()
iface_pd     = f_iface.result()
rt_rpc_pd    = f_rt_rpc.result()
cg_ts_pd     = f_cg_ts.result()
rt_out_pd    = f_rt_out.result()


#### RPC Type Distribution

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("MSCallGraph — RPC Type Distribution\n",
             color=PALETTE["accent"])

ax = axes[0]
colors_rpc = sns.color_palette("tab10", len(rpctype_pd))
wedges, texts, autotexts = ax.pie(
    rpctype_pd["count"], labels=[t if t != "userDefined" else "other" for t in rpctype_pd["rpctype"]],
    colors=colors_rpc, autopct="%1.1f%%", startangle=140,
    wedgeprops=dict(edgecolor=PALETTE["bg"], linewidth=2),
    textprops={"color": PALETTE["text"]},
)
ax.set_title("Call Volume by RPC Type")
#ax.set_yticklabels([t if t != "userDefined" else "other" for t in rpctype_pd["rpctype"]], fontsize=9)

ax2 = axes[1]
ax2.barh([t if t != "userDefined" else "other" for t in rpctype_pd["rpctype"]], rpctype_pd["count"],
         color=colors_rpc, edgecolor=PALETTE["bg"])
ax2.set_xlabel("Call Count")
ax2.set_title("Absolute Call Volume per Type")
save(fig,"RPC_Type_distribution.png")
display(fig)

#### Response Time histogram

In [0]:
# ── D2. Response Time histogram ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle("MSCallGraph — Response Time (RT) Distribution",
             color=PALETTE["accent"])
ax.bar(rt_hist_pd["bin_center"], rt_hist_pd["count"],
       width=(rt_hi - rt_lo) / len(rt_hist_pd) * 0.9,
       color=PALETTE["rt"], alpha=0.85, edgecolor=PALETTE["bg"])
ax.set_xlabel("Response Time (ms)")
ax.set_ylabel("Count")
ax.set_title("RT Distribution (log-scale y)")
ax.set_yscale("log")
save(fig,"tr_distribution.png")
display(fig)

#### Top edges heatmap (UM × DM call volume)

In [0]:
# ── D3. Top edges heatmap (UM × DM call volume) ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("MSCallGraph — Top 30 UM→DM Edges\n"
             "Direct blueprint for GNN edge construction",
             color=PALETTE["accent"])

# Call count heatmap
pivot_count = (top_edges_pd.pivot_table(
    index="UM", columns="DM", values="call_count", fill_value=0
))
ax = axes[0]
sns.heatmap(pivot_count, ax=ax, cmap="YlOrRd", linewidths=0.3,
            cbar_kws={"label": "Call Count"},
            annot=(pivot_count.shape[0] <= 15), fmt=".0f", annot_kws={"size": 6})
ax.set_title("Call Count (edge weight)")
ax.tick_params(labelsize=7)

# Mean RT heatmap
pivot_rt = (top_edges_pd.pivot_table(
    index="UM", columns="DM", values="mean_rt", fill_value=0
))
ax2 = axes[1]
sns.heatmap(pivot_rt, ax=ax2, cmap="Blues", linewidths=0.3,
            cbar_kws={"label": "Mean RT (ms)"},
            annot=(pivot_rt.shape[0] <= 15), fmt=".1f", annot_kws={"size": 6})
ax2.set_title("Mean Response Time (edge latency)")
ax2.tick_params(labelsize=7)
save(fig,"Top_edges_heatmap.png")
display(fig)

#### Node degree distribution

In [0]:
# ── D4. Node degree distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("MSCallGraph — Node In/Out Degree Distribution\n",
             color=PALETTE["accent"])

for ax, col, color, label in [
    (axes[0], "out_degree", PALETTE["traffic"], "Out-Degree"),
    (axes[1], "in_degree",  PALETTE["callgraph"], "In-Degree"),
]:
    vals = node_deg_pd[col].values
    bins = np.linspace(0, np.percentile(vals, 99), 40)
    ax.hist(vals, bins=bins, color=color, alpha=0.85, edgecolor=PALETTE["bg"])
    ax.set_xlabel(label); ax.set_ylabel("# Nodes")
    ax.set_title(f"{label} Distribution")
    ax.set_yscale("log")
    ax.axvline(np.median(vals), color="white", lw=1.2, linestyle="--",
               label=f"median={np.median(vals):.0f}")
    ax.legend(fontsize=8, facecolor=PALETTE["grid"], labelcolor=PALETTE["text"])
save(fig,"IN_OUT_degree_distribution.png")
display(fig)

#### RT Spread per RPC Type

In [0]:
# ── D6. RT boxplot by rpctype ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle("RT Spread per RPC Type\n",
             color=PALETTE["accent"])

types  = rt_rpc_pd["rpctype"].tolist()
colors_b = sns.color_palette("tab10", len(types))

for i, (_, row) in enumerate(rt_rpc_pd.iterrows()):
    pct = row["pct"]
    if pct is None or len(pct) < 4:
        continue
    q1, med, q3, p95 = pct[0], pct[1], pct[2], pct[3]
    # Draw manual box
    ax.broken_barh([(q1, q3 - q1)], (i - 0.3, 0.6),
                   facecolors=colors_b[i], alpha=0.6)
    ax.plot([med, med], [i - 0.3, i + 0.3], color="white", lw=2)
    ax.plot([q3, p95], [i, i], color=colors_b[i], lw=1.5)
    ax.scatter([row["mean"]], [i], color="white", s=25, zorder=5)
    ax.text(p95 * 1.02, i, f"n={int(row['n']):,}", fontsize=7,
            color=PALETTE["text"], va="center")

ax.set_yticks(range(len(types)))
ax.set_yticklabels([t if t != "userDefined" else "other" for t in types], fontsize=9)
ax.set_xlabel("Response Time (ms)")
ax.legend([
    "IQR (box)", 
    "95th percentile (whisker)", 
    "Mean (white dot)"
], fontsize=8, facecolor=PALETTE["grid"], labelcolor=PALETTE["text"], loc="upper right")
save(fig, "RT_Spread_per_RPC_Type.png")
display(fig)


In [0]:
# ── D7. Interface-level RT (top 20) ───────────────────────────────────────────
if not iface_pd.empty:
    fig, ax = plt.subplots(figsize=(14, 7))
    fig.suptitle("MSCallGraph — Top 20 Interfaces by RT\n"
                 "Interface = micro edge type for heterogeneous GNN",
                 color=PALETTE["accent"])
    iface_pd = iface_pd.sort_values("mean_rt", ascending=True)
    y = np.arange(len(iface_pd))
    pcts_iface = np.array([list(p) for p in iface_pd["pct_rt"]])
    ax.barh(y, pcts_iface[:, 2] - pcts_iface[:, 0], left=pcts_iface[:, 0],
            color=PALETTE["rt"], alpha=0.6, label="IQR")
    ax.scatter(iface_pd["mean_rt"], y, color="white", s=30, zorder=5, label="Mean RT")
    ax.scatter(pcts_iface[:, 1], y, color=PALETTE["accent"], s=20, marker="|",
               zorder=5, label="Median RT")
    ax.set_yticks(y)
    ax.set_yticklabels(
        [str(v)[:40] for v in iface_pd["interface"].tolist()], fontsize=7
    )
    ax.set_xlabel("Response Time (ms)")
    ax.legend(fontsize=8, facecolor=PALETTE["grid"], labelcolor=PALETTE["text"])
    display(fig)

In [0]:
# ── D9. RT Outlier rate over time ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 4))
fig.suptitle("MSCallGraph — RT Outlier Rate (IQR k=3)\n"
             "Spike windows → candidate anomaly timestamps for GNN labels",
             color=PALETTE["accent"])
ax.fill_between(rt_out_pd["t_idx"], rt_out_pd["rate"]*100, color=PALETTE["rt"], alpha=0.5)
ax.plot(rt_out_pd["t_idx"], rt_out_pd["rate"]*100, color=PALETTE["rt"], lw=1.2)
ax.set_xlabel("t_idx"); ax.set_ylabel("Outlier Rate (%)")
display(fig)


## SECTION E — CROSS-TABLE: GNN Node & Edge Feature Summary

In [0]:
def _cross_join_ts():
    """Merge all three temporal aggregations for cross-correlation."""
    res_ts = (res_df.groupBy("t_idx")
              .agg(F.mean("cpu_utilization").alias("cpu"),
                   F.mean("memory_utilization").alias("memory"))
              .orderBy("t_idx").toPandas())
    cg_ts2 = (cg_df.groupBy("t_idx")
              .agg(F.mean("rt").alias("rt"))
              .orderBy("t_idx").toPandas())
    merged = pd.merge(res_ts, cg_ts2, on="t_idx", how="inner")
    if not rtq_ts_pd.empty and TRAFFIC_COLS:
        rtq_sub = rtq_ts_pd[["t_idx"] + [c for c in TRAFFIC_COLS_MCR if c in rtq_ts_pd.columns]].copy()
        merged  = pd.merge(merged, rtq_sub, on="t_idx", how="left")
    return merged

cross_pd = _cross_join_ts()

In [0]:
# ── E1. Cross-table feature heatmap ───────────────────────────────────────────
numeric_cols = [c for c in cross_pd.columns if c != "t_idx" and cross_pd[c].dtype in [float, int, "float64","int64"]]
if len(numeric_cols) > 1:
    corr_cross = cross_pd[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    fig.suptitle("Feature Correlation (CPU, Memory, RT, Calls, MCR)\n",color=PALETTE["accent"])
    mask = np.triu(np.ones_like(corr_cross, dtype=bool))
    sns.heatmap(corr_cross, ax=ax, cmap="coolwarm", center=0,
                annot=True, fmt=".2f", annot_kws={"size": 8},
                linewidths=0.5, mask=mask,
                cbar_kws={"label": "Pearson r"})
    ax.set_title(" Pearson Correlation")
    save(fig,"E1.png")
    display(fig)

### E2. Joint temporal scatter matrix (pairplot proxy) 

In [0]:
if len(numeric_cols) >= 3:
    plot_cols = numeric_cols[:min(5, len(numeric_cols))]
    n = len(plot_cols)
    fig, axes = plt.subplots(n, n, figsize=(4*n, 4*n))
    fig.suptitle("Cross-Table Scatter Matrix — Temporal Index Aligned\n"
                 "Each point = one t_idx snapshot",
                 color=PALETTE["accent"], fontsize=13)

    for i, ci in enumerate(plot_cols):
        for j, cj in enumerate(plot_cols):
            ax = axes[i][j]
            if i == j:
                vals = cross_pd[ci].dropna().values
                ax.hist(vals, bins=30, color=PALETTE["accent"], alpha=0.8)
                ax.set_title(ci, fontsize=8, color=PALETTE["accent"])
            elif i > j:
                ax.scatter(cross_pd[cj], cross_pd[ci],
                           alpha=0.4, s=8, color=PALETTE["traffic"])
                r = cross_pd[[ci, cj]].dropna().corr().iloc[0, 1]
                ax.text(0.05, 0.95, f"r={r:.2f}", transform=ax.transAxes,
                        fontsize=8, color=PALETTE["accent"], va="top")
            else:
                ax.axis("off")
            if i == n-1: ax.set_xlabel(cj, fontsize=7)
            if j == 0:   ax.set_ylabel(ci, fontsize=7)
            ax.tick_params(labelsize=6)

    plt.tight_layout()
    display(fig)

In [0]:
# ── E3. GNN Design Summary Card ───────────────────────────────────────────────
print("\n[E3] Generating GNN design recommendation card …")

fig = plt.figure(figsize=(14, 10))
fig.patch.set_facecolor(PALETTE["bg"])
ax = fig.add_axes([0, 0, 1, 1])
ax.set_facecolor(PALETTE["bg"])
ax.axis("off")

lines = [
    ("TEMPORAL GNN DESIGN GUIDE", 0.96, 16, PALETTE["accent"], "bold"),
    ("based on Alibaba Cloud 2021 Microservices Trace EDA", 0.92, 10, PALETTE["text"], "normal"),

    ("─── NODES ─────────────────────────────────────────────────", 0.87, 9, PALETTE["grid"], "normal"),
    ("Source: MSResource + MSRTQps  (join on msname/t_idx)", 0.83, 10, PALETTE["text"], "normal"),
    ("Node features:  [cpu_utilization, memory_utilization,", 0.79, 10, PALETTE["text"], "normal"),
    ("                providerRPC_MCR, providerRPC_RT,",        0.75, 10, PALETTE["text"], "normal"),
    ("                HTTP_MCR, HTTP_RT, consumerMQ_MCR, …]",   0.71, 10, PALETTE["text"], "normal"),
    ("Node identity:  unique msname (microservice)",            0.67, 10, PALETTE["neutral"], "normal"),

    ("─── EDGES ─────────────────────────────────────────────────", 0.62, 9, PALETTE["grid"], "normal"),
    ("Source: MSCallGraph  (UM → DM per rpctype)",              0.58, 10, PALETTE["text"], "normal"),
    ("Edge features:  [call_count, mean_rt, p95_rt, rpctype]",  0.54, 10, PALETTE["text"], "normal"),
    ("Edge type:      rpctype ∈ {http, rpc, mq, …}",            0.50, 10, PALETTE["neutral"], "normal"),
    ("Edge weight:    call_count or 1/mean_rt (latency-aware)", 0.46, 10, PALETTE["neutral"], "normal"),

    ("─── TEMPORAL SNAPSHOTS ────────────────────────────────────", 0.41, 9, PALETTE["grid"], "normal"),
    ("Snapshot key:   t_idx (aligned across all 3 tables)",     0.37, 10, PALETTE["text"], "normal"),
    ("Recommended K:  look at ACF dominant lag → temporal window size", 0.33, 10, PALETTE["text"], "normal"),
    ("Seasonality:    Check ACF peaks (e.g. 24-step = 1 day)",  0.29, 10, PALETTE["neutral"], "normal"),

    ("─── PREDICTION TARGETS ────────────────────────────────────", 0.24, 9, PALETTE["grid"], "normal"),
    ("  • CPU / Memory utilisation  (regression)",               0.20, 10, PALETTE["cpu"], "normal"),
    ("  • Traffic MCR / RT          (regression)",               0.16, 10, PALETTE["mcr"], "normal"),
    ("  • Anomaly / outlier flag    (classification)",           0.12, 10, PALETTE["rt"],  "normal"),
    ("  • Load imbalance score      (node-level regression)",    0.08, 10, PALETTE["traffic"], "normal"),
]

for text, y, size, color, weight in lines:
    ax.text(0.05, y, text, transform=ax.transAxes,
            fontsize=size, color=color, fontweight=weight,
            fontfamily="monospace", va="top")

save(fig, "E3_gnn_design_summary.png")


# ═══════════════════════════════════════════════════════════════════════════════
#  SECTION F — SEASONALITY DEEP DIVE (all 3 tables)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[F] Seasonality deep dive …")

def _hour_profile(df, agg_col, alias, time_granularity=24):
    """Cyclic aggregation: group t_idx mod granularity."""
    return (df
        .withColumn("hour_of_period", F.col("t_idx") % time_granularity)
        .groupBy("hour_of_period")
        .agg(F.mean(agg_col).alias(alias))
        .orderBy("hour_of_period")
        .toPandas()
    )

with ThreadPoolExecutor(max_workers=3) as pool:
    f_cpu_hour  = pool.submit(_hour_profile, res_df,  "cpu_utilization",    "cpu", 24)
    f_mem_hour  = pool.submit(_hour_profile, res_df,  "memory_utilization", "mem", 24)
    f_rt_hour   = pool.submit(_hour_profile, cg_df,   "rt",                 "rt",  24)

cpu_hour = f_cpu_hour.result()
mem_hour = f_mem_hour.result()
rt_hour  = f_rt_hour.result()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), subplot_kw=dict(polar=False))
fig.suptitle("Diurnal Seasonality Profile (t_idx mod 24)\n"
             "Indicates daily periodicity — set GNN temporal K accordingly",
             color=PALETTE["accent"])

for ax, df, col, color, label in [
    (axes[0], cpu_hour, "cpu", PALETTE["cpu"],    "CPU Utilization"),
    (axes[1], mem_hour, "mem", PALETTE["memory"], "Memory Utilization"),
    (axes[2], rt_hour,  "rt",  PALETTE["rt"],     "Mean RT (ms)"),
]:
    ax.plot(df["hour_of_period"], df[col], color=color, lw=2, marker="o", ms=4)
    ax.fill_between(df["hour_of_period"], df[col], alpha=0.2, color=color)
    ax.set_xlabel("t_idx mod 24 (hour proxy)")
    ax.set_ylabel(label)
    ax.set_title(f"Daily Pattern — {label}")
    # highlight peak
    pk = df.loc[df[col].idxmax()]
    ax.annotate(f"peak@{int(pk['hour_of_period'])}",
                (pk["hour_of_period"], pk[col]),
                xytext=(5, 5), textcoords="offset points",
                color="white", fontsize=8)

display(fig)